# 13. Global programs

A **global program** is a Python function on the fields of a logical mesh (see "12. Logical meshes"), with global semantics: it means a function of the whole mesh. Its source has no ranks, no halos and no communication. Run directly, on one process, it is the reference. **Lowered** onto a distribution of the mesh, it becomes a local program on every rank (see "10. Local programs"): the lowering places the exchanges and the global sums. The **simulation condition**: the lowered run gives the same results as the direct one, bitwise.

```
@program()    --decorates a Python function-->  Program            (run directly: the reference)
Program       --lower(distribution)-->          LoweredProgram     (one local program per rank)
LoweredProgram --called, run_local on the ranks-->  the results, gathered: the same as the direct run
```

The example: mass sharing on the plate of 4 × 2 cells of notebook 10, the same computation, now without `Owned` and `exchange` in the source.

Contents: programs · what programs cannot do · distributions · the lowering · the simulation condition · coverage · reproducible sums and the compile key · local and global.

## Programs

| Concept | Kind | Meaning |
|---|---|---|
| `program` | decorator | `@program(backend=...)` makes a `Program` of a Python function, after checking it with `PROGRAM_RULES`; its DSL operators run on `backend` (embedded if `None`) |
| `Program` | class | Called with mesh fields of one logical mesh, it runs the function directly; `operators` are the DSL operators its body calls by name; `lower(distribution)` gives a `LoweredProgram` |
| `PROGRAM_RULES` | `DSLRules` | Fields, `int`, `float`, tuples and dataclasses as arguments (of non-field values; see below); expressions, assignments, `return`, `if` and `for` loops; no attributes; `domain=` is allowed (`top_level`); no names of the decomposition or field constructors (see below) |
| `program_mesh` | context variable | The logical mesh of the running program, while its body runs |

In a running program:
- `region(East)` is the **global** domain of the region, from the program's logical mesh, e.g. for `op[region(East)](...)`;
- `sum_over(f, axis=Cell)`, over a `Location`, is a **global sum** of a field with one dimension (`domain=` restricts it): the values added one by one in the order of the global indices, so the same sum for any decomposition. It is only legal at program level, never in field operators;
- DSL operators are called as anywhere: their results are mesh fields of the program's mesh.

In [1]:
import contextlib
import io
import math

import numpy as np
from concepts import (
    MISSING_VALUE,
    PROGRAM_RULES,
    Axis,
    Dims,
    Face,
    Field,
    LogicalMesh,
    Neighbor,
    Region,
    Relation,
    Schema,
    region,
)
from implementation import field_operator, neighbor_table, program, sum_over


class Plate(Schema): ...


Cell = Face[Plate]


class K(Axis): ...  # vertical levels


class C2C(Relation[Cell, Cell], max_neighbors=4, partial=True): ...  # [west, east, south, north]


class West(Region[Cell]): ...


class East(Region[Cell]): ...


def at(i, j):  # the cell at column i, row j, or missing
    return 4 * j + i if 0 <= i < 4 and 0 <= j < 2 else MISSING_VALUE


rows = [
    [at(i - 1, j), at(i + 1, j), at(i, j - 1), at(i, j + 1)] for j in range(2) for i in range(4)
]
grid = LogicalMesh(
    Plate,
    {C2C: neighbor_table(C2C, rows)},
    extents={Cell: 8, K: 2},
    regions={West: (0, 4), East: (3, 7)},
)


@field_operator()
def share(
    m: Field[Dims[Cell], float], one: Field[Dims[Cell], float], k: float
) -> Field[Dims[Cell], float]:
    received = sum_over(m(C2C.boundary(0.0)), axis=Neighbor[C2C])  # the neighbors' mass
    given = sum_over(one(C2C.boundary(0.0)), axis=Neighbor[C2C]) * m  # to each neighbor
    return m + k * (received - given)


@program()
def sharing(
    m: Field[Dims[Cell], float], one: Field[Dims[Cell], float], k: float, steps: int
) -> tuple[Field[Dims[Cell], float], float, float]:
    for _ in range(steps):
        m = share(m, one, k)
    return m, sum_over(m, axis=Cell), sum_over(m, axis=Cell, domain=region(East))


m0 = grid.field(Cell, data=[8.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])  # all the mass in cell 0
ones = grid.full(Cell, fill_value=1.0)
print(
    "operators:", [op.__name__ for op in sharing.operators], "| top level:", PROGRAM_RULES.top_level
)
m3, total_mass, east_mass = sharing(m0, ones, 0.125, 3)
# as in notebook 10: cell 0: 8, 6, then 6 + (1 + 1 - 2 * 6) / 8 = 4.75, then
# 4.75 + (1.375 + 1.5 - 2 * 4.75) / 8 = 3.921875; the mass is conserved; East: cells 3 and 7
print("m =", m3)
print("total:", total_mass, "| East:", east_mass, "| m.mesh is grid:", m3.mesh is grid)

operators: ['share'] | top level: True
m = Field(Face[Plate][0:8]: [3.921875, 1.5, 0.25, 0.015625, 1.75, 0.515625, 0.046875, 0.0])
total: 8.0 | East: 0.015625 | m.mesh is grid: True


## What programs cannot do

Programs have global semantics: lowered, each rank runs the same body on its own part of the mesh, with local fields. So:

- **No decomposition** (rejected by `PROGRAM_RULES`): `Owned`, `exchange`, `global_sum`, `current_mesh`, `run_local`, `local_program`, `scatter`. Regions are requested with `region(...)`.
- **No field constructors** (rejected by `PROGRAM_RULES`): `NumPyArrayField`, `DeferredField`, `mesh_field`, `gather_owned`. Fields come in as arguments.
- **No attributes**, e.g. `t.data`: programs do not look inside fields.
- **Fields are separate arguments**, checked at run time in the direct and the lowered run alike (a `TypeError`):
  - a field from a global or a closure, e.g. a geometry field, given to an operator, summed, combined or returned: pass it as an argument;
  - a field inside a tuple, list, dict or dataclass argument, of the program or of an operator it calls;
  - results are fields, scalars, or tuples of them.
- **Field operations at program level** are only pointwise arithmetic, e.g. `m + 2.0 * m`. Premaps, subscripts, writes, neighbor reductions, `flatten` and `unflatten` raise a `TypeError` (the guard): lowered, they would read a rank's local field. Compute with fields in field operators.
- **No scans along a `Location`** (a `NotImplementedError`): a scan along a distributed dimension needs a communication algorithm. Scans along an `Axis`, e.g. `K`, are fine.
- `sum_over` over a `Location` only at program level: in a field operator it raises a `TypeError`, as outside programs.
- Plain helper functions called by a program run at program level too, under the same checks; they are not checked statically: a helper reading `.data` could make the control flow differ between ranks.

In [2]:
from concepts import Owned
from implementation import exchange, max_over


def exchanging(m: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    return exchange(m)


def owned_only(m: Field[Dims[Cell], float]) -> float:
    return sum_over(m, axis=Cell, domain=Owned(Cell))


def looking_inside(m: Field[Dims[Cell], float]) -> float:
    return m.data[0]


for definition in (exchanging, owned_only, looking_inside):  # rejected by `PROGRAM_RULES`
    try:
        program()(definition)
    except TypeError as error:
        print("TypeError:", error)


@program()
def premapped(
    m: Field[Dims[Cell], float], one: Field[Dims[Cell], float]
) -> Field[Dims[Cell], float]:
    shared = share(m, one, 0.125)
    return max_over(shared(C2C), axis=Neighbor[C2C])  # a premap at program level


@program()
def subscripted(
    m: Field[Dims[Cell], float], one: Field[Dims[Cell], float]
) -> Field[Dims[Cell], float]:
    return share(m, one, 0.125)[region(East)]  # a subscript at program level


@program()
def tripled(m: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    return m + 2.0 * m  # pointwise arithmetic: allowed


@field_operator()
def mass(m: Field[Dims[Cell], float]) -> float:
    # a global sum in a field operator, even one called by a program: field operators are L1
    # code, so the error says to call it in a `@program` directly
    return sum_over(m, axis=Cell)


@program()
def operator_sum(m: Field[Dims[Cell], float]) -> float:
    return mass(m)


AREA = grid.field(Cell, data=[1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 2.0, 2.0])  # a global field


@program()
def weighted(m: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    return m * AREA  # not an argument: pass it as one


@program()
def in_a_tuple(m: Field[Dims[Cell], float], pair: tuple) -> Field[Dims[Cell], float]:
    return m


other = LogicalMesh(Plate, {C2C: grid[C2C]}, extents={Cell: 8})  # another logical mesh
for run in (
    lambda: premapped(m0, ones),
    lambda: subscripted(m0, ones),
    lambda: sum_over(m0, axis=Cell),  # outside a program
    lambda: operator_sum(m0),
    lambda: weighted(m0),
    lambda: in_a_tuple(m0, (m0, ones)),
    lambda: sharing(m0, other.full(Cell, fill_value=1.0), 0.125, 3),
):
    try:
        run()
    except (TypeError, ValueError) as error:
        print(f"{type(error).__name__}: {error}")
print("m + 2.0 * m =", tripled(m0))  # cell 0: 8 + 16 = 24

TypeError: 'exchanging' is not a valid program definition: uses the collective 'exchange'.
TypeError: 'owned_only' is not a valid program definition: uses the forbidden name 'Owned'.
TypeError: 'looking_inside' is not a valid program definition: the syntax `Attribute` is not accepted.
TypeError: A premap in a program: compute with fields in field operators (programs call operators, `sum_over` over a `Location`, and pointwise arithmetic).
TypeError: A subscript in a program: compute with fields in field operators (programs call operators, `sum_over` over a `Location`, and pointwise arithmetic).
TypeError: `sum_over` sums over a neighbor dimension, `Neighbor[R]`, not 'Face[Plate]'. Over a `Location`, call it in a `@program` (global semantics), or `global_sum` in a local program.
TypeError: `sum_over` sums over a neighbor dimension, `Neighbor[R]`, not 'Face[Plate]'. Over a `Location`, call it in a `@program` (global semantics), or `global_sum` in a local program.
TypeError: Programs get t

## Distributions

| Concept | Kind | Meaning |
|---|---|---|
| `Distribution` | dataclass | How a logical mesh is distributed: its `mesh` and the `LocalMesh` of each rank, `local_meshes`. Configuration, never in the source of programs, and not part of the mesh's identity: the local meshes carry the `mesh_id` of the logical mesh |
| `distribute(mesh, owner, support=..., hops=...)` | function | The *given* mode: each rank owns the points `owner` gives it, and its halo is the points reached through the relations of `support`, `hops` times (as `decompose` in 10) |
| `derive_distribution(mesh, owner, programs)` | function | The *derived* mode: each rank's halo is exactly what the reads of the programs' operators need for its owned points |
| `footprints(operators, mesh, owned)` | function | Per rank, the global points the reads of the operators need for the owned points, the points read on the way included |
| `check_coverage(operators, distribution)` | function | A `ValueError` if a rank's local points do not hold the footprint of its owned points |

A rank may own nothing: it computes empty fields, and takes part in the exchanges and sums.

In [3]:
from implementation import check_coverage, derive_distribution, distribute, footprints

HALVES = {Cell: [0, 0, 1, 1, 0, 0, 1, 1]}  # the columns 0-1 and 2-3, as in notebook 10
single = distribute(grid, {Cell: [0] * 8}, support=(C2C,))
halves = distribute(grid, HALVES, support=(C2C,))
thirds = distribute(grid, {Cell: [0, 1, 1, 2, 0, 1, 1, 2]}, support=(C2C,))  # 0, 1-2, 3
derived = derive_distribution(grid, owner=HALVES, programs=[sharing])  # from the reads of `share`


def indices(distribution):  # the global indices of the local points of each rank
    return [local_mesh.global_index[Cell] for local_mesh in distribution.local_meshes]


# owned first, then the halo, each in global order, e.g. halves, rank 0: 0, 1, 4, 5, then 2, 6
for name, distribution in {"halves": halves, "thirds": thirds, "derived": derived}.items():
    print(f"{name:<8}", indices(distribution))
print(
    "derived == halves:",
    indices(derived) == indices(halves),
    "| mesh_id:",
    halves.local_meshes[0].mesh_id,
)
# `share` reads the 4 neighbors of each owned cell: rank 0 needs 2 and 6, rank 1 needs 1 and 5
needed = footprints(sharing.operators, grid, [{Cell: (0, 1, 4, 5)}, {Cell: (2, 3, 6, 7)}])
print("footprints:", [sorted(points[Cell]) for points in needed])
try:
    check_coverage(sharing.operators, distribute(grid, HALVES, support=(C2C,), hops=0))
except ValueError as error:
    print("ValueError:", error)
idle = distribute(grid, {Cell: [1] * 8}, support=(C2C,))  # rank 0 owns nothing
print("rank 0 owns nothing:", indices(idle))

halves   [(0, 1, 4, 5, 2, 6), (2, 3, 6, 7, 1, 5)]
thirds   [(0, 4, 1, 5), (1, 2, 5, 6, 0, 3, 4, 7), (3, 7, 2, 6)]
derived  [(0, 1, 4, 5, 2, 6), (2, 3, 6, 7, 1, 5)]
derived == halves: True | mesh_id: Plate-0
footprints: [[0, 1, 2, 4, 5, 6], [1, 2, 3, 5, 6, 7]]
ValueError: The distribution of 'Plate-0' does not cover the reads of 'share': rank 0 needs Face[Plate]{2, 6} (distribute with more `hops`, or derive the distribution).
rank 0 owns nothing: [(), (0, 1, 2, 3, 4, 5, 6, 7)]


## The lowering

`sharing.lower(distribution)` checks the coverage of the program's operators (before any execution) and returns a `LoweredProgram`. Called with mesh fields, it gives each rank its owned values, runs the body on every rank as a local program, and gathers the results. On each rank, the body runs as written, with:

- **exchanges** placed before each operator call, for each argument the operator reads through a relation, unless the value is fresh: already exchanged, e.g. `one` after its first exchange;
- **results** computed on the owned points (or the owned points of a requested `region(...)`);
- **global sums**: `sum_over(f, axis=Cell)` becomes `global_sum` on the owned points;
- the **log**, `lowered.log`: the actions of rank 0 in the last call, the plan a compiler would generate.

The decisions depend only on the operators and the control flow, the same on every rank, so all the ranks call the same collectives. With one rank, there are no exchanges.

In [4]:
def same(a, b):  # bitwise the same results: fields (domains, values, mesh_id), tuples, scalars
    if isinstance(a, tuple):
        return isinstance(b, tuple) and len(a) == len(b) and all(map(same, a, b))
    if isinstance(a, Field):
        return (a.domain, a.data.tolist(), a.mesh.mesh_id) == (
            b.domain,
            b.data.tolist(),
            b.mesh.mesh_id,
        )
    return type(a) is type(b) and a == b


lowered = sharing.lower(halves)
direct = sharing(m0, ones, 0.125, 3)
with contextlib.redirect_stdout(io.StringIO()):  # silence the backends, on all the threads
    results = lowered(m0, ones, 0.125, 3)
print("lowered == direct:", same(results, direct), "| m =", results[0].data.tolist())
# `one` is exchanged once, `m` before each step; then the total and the East sum
print("log on halves:", *lowered.log, sep="\n  ")
on_single = sharing.lower(single)
on_single(m0, ones, 0.125, 3)
print("log on single:", *on_single.log, sep="\n  ")  # one rank: no exchanges

lowered == direct: True | m = [3.921875, 1.5, 0.25, 0.015625, 1.75, 0.515625, 0.046875, 0.0]
log on halves:
  exchange 'm' for 'share'
  exchange 'one' for 'share'
  compute 'share' on the owned points
  exchange 'm' for 'share'
  compute 'share' on the owned points
  exchange 'm' for 'share'
  compute 'share' on the owned points
  global_sum over 'Face[Plate]'
  global_sum over 'Face[Plate]'
log on single:
  compute 'share' on the owned points
  compute 'share' on the owned points
  compute 'share' on the owned points
  global_sum over 'Face[Plate]'
  global_sum over 'Face[Plate]'


## The simulation condition

The lowered run is correct if it gives the results of the direct run, for every distribution and backend: the same domains and the same values, **bitwise** (`==` on floats), and the same scalars. The global sums add the values in the order of the global indices, so they are bitwise the same too.

Below, from `ms`, the square roots `sqrt(c + 1)`: values that are not dyadic fractions, so that a different order of the operations would show. Then `mixed`, with a scan along `K`, a nested operator, and an `if` on a global sum inside the loop (the same branch on every rank: the sum is the same), and pointwise arithmetic.

In [5]:
from implementation import column_major, compiled, eager, lazy, lazy_backward, scan_operator


@field_operator()
def around_k(t: Field[Dims[Cell, K], float]) -> Field[Dims[Cell, K], float]:
    return sum_over(t(C2C.boundary(0.0)), axis=Neighbor[C2C])


@field_operator()
def smooth_k(t: Field[Dims[Cell, K], float]) -> Field[Dims[Cell, K], float]:
    return 0.5 * t + 0.125 * around_k(t)  # a nested operator


@scan_operator(axis=K)
def colsum(carry: float, x: float) -> float:
    return carry + x


@program()
def mixed(
    m: Field[Dims[Cell], float],
    one: Field[Dims[Cell], float],
    c: Field[Dims[Cell, K], float],
    k: float,
    steps: int,
) -> tuple[Field[Dims[Cell], float], Field[Dims[Cell, K], float], float, Field[Dims[Cell], float]]:
    total = 0.0
    for _ in range(steps):
        m = share(m, one, k)
        total = sum_over(m, axis=Cell, domain=region(East))
        if total > 0.01:  # the East totals: 0, 0, 0.015625, 0.05078125
            c = smooth_k(c)
        else:
            c = colsum(c)
    return m, c, total, m + 2.0 * m


ms = grid.field(Cell, data=[math.sqrt(c + 1.0) for c in range(8)])
cs = grid.field(Cell, K, data=np.sqrt(np.arange(16.0) + 1).reshape(8, 2))
runs = {"sharing": (sharing, (ms, ones, 0.125, 4)), "mixed": (mixed, (m0, ones, cs, 0.125, 4))}
for backend in (None, eager, lazy, lazy_backward, column_major, compiled):
    for name, (embedded, args) in runs.items():
        p = program(backend=backend)(embedded.function)  # the same function, on `backend`
        distributions = (single, halves, thirds, derive_distribution(grid, HALVES, [p]))
        with contextlib.redirect_stdout(io.StringIO()):  # silence the backends
            direct = p(*args)
            equal = [same(p.lower(d)(*args), direct) for d in distributions]
        print(f"{str(backend or 'embedded'):<14} {name:<8} single, halves, thirds, derived:", equal)
print("mixed: the last East total =", mixed(m0, ones, cs, 0.125, 4)[2])

embedded       sharing  single, halves, thirds, derived: [True, True, True, True]


embedded       mixed    single, halves, thirds, derived: [True, True, True, True]
eager          sharing  single, halves, thirds, derived: [True, True, True, True]


eager          mixed    single, halves, thirds, derived: [True, True, True, True]
lazy           sharing  single, halves, thirds, derived: [True, True, True, True]


lazy           mixed    single, halves, thirds, derived: [True, True, True, True]
lazy_backward  sharing  single, halves, thirds, derived: [True, True, True, True]


lazy_backward  mixed    single, halves, thirds, derived: [True, True, True, True]
column_major   sharing  single, halves, thirds, derived: [True, True, True, True]


column_major   mixed    single, halves, thirds, derived: [True, True, True, True]
compiled       sharing  single, halves, thirds, derived: [True, True, True, True]


compiled       mixed    single, halves, thirds, derived: [True, True, True, True]
mixed: the last East total = 0.05078125


## Coverage

A distribution must hold, on each rank, the footprint of its owned points: the points the reads of the operators need, through all the steps of a chain of reads. Else the results would silently shrink. It is checked twice:

- **before execution**, by `lower`: `check_coverage` on the operators the body calls by name, `Program.operators`;
- **at each call**, on each rank: the local reads from the owned points must stay within the local points (a `ValueError` "beyond the local support"). This also covers operators called through plain helper functions, which `operators` does not find.

Below, a strip of 8 cells and `hop2`, which reads two hops away: a halo of one hop is not enough, the derived distribution is.

In [6]:
class Strip(Schema): ...


SCell = Face[Strip]


class L2R(Relation[SCell, SCell], max_neighbors=2, partial=True): ...  # [left, right]


M = MISSING_VALUE
line_rows = [[c - 1 if c > 0 else M, c + 1 if c < 7 else M] for c in range(8)]
line = LogicalMesh(Strip, {L2R: neighbor_table(L2R, line_rows)}, extents={SCell: 8})


@field_operator()
def hop2(t: Field[Dims[SCell], float]) -> Field[Dims[SCell], float]:  # two hops: SCell[2:6]
    inner = sum_over(t(L2R)(L2R), axis=Neighbor[L2R, Neighbor[L2R]])
    return sum_over(inner, axis=Neighbor[L2R])


@program()
def two(t: Field[Dims[SCell], float]) -> Field[Dims[SCell], float]:
    return hop2(t)


strip_owner = {SCell: [0, 0, 0, 1, 1, 1, 2, 2]}
try:
    two.lower(distribute(line, strip_owner, support=(L2R,), hops=1))  # before any execution
except ValueError as error:
    print("ValueError:", error)  # 'Strip-2': the third logical mesh of this kernel
line_derived = derive_distribution(line, strip_owner, [two])
print("derived:", [lm.global_index[SCell] for lm in line_derived.local_meshes])  # 2 hops each way
squares = line.field(SCell, data=[float(c * c) for c in range(8)])
# at c: the neighbors of c - 1 and c + 1: (c - 2)^2 + c^2 + c^2 + (c + 2)^2 = 4 c^2 + 8
direct = two(squares)
print("direct:", direct, "| lowered == direct:", same(two.lower(line_derived)(squares), direct))

ValueError: The distribution of 'Strip-2' does not cover the reads of 'hop2': rank 0 needs Face[Strip][4:5] (distribute with more `hops`, or derive the distribution).
derived: [(0, 1, 2, 3, 4), (3, 4, 5, 1, 2, 6, 7), (6, 7, 4, 5)]
direct: Field(Face[Strip][2:6]: [24.0, 44.0, 72.0, 108.0]) | lowered == direct: True


## Reproducible sums and the compile key

A global `sum_over` is reproducible, not exact: the same sum, bitwise, for any decomposition and in the direct run, as `global_sum` in notebook 10.

A program builds the operators of its backend once per operator and **mesh signature** (see "12. Logical meshes"): the schema, the relations and the regions, not the mesh or its values. So `compiled` compiles `share` once, for the direct run, every distribution, and every mesh of the same signature.

In [7]:
@program()
def total(v: Field[Dims[Cell], float]) -> float:
    return sum_over(v, axis=Cell)


v = grid.field(Cell, data=[1.0, 1e16, 1.0, -1e16, 1.0, 1.0, 1.0, 1.0])
# in global order: 1 + 1e16 = 1e16 (the 1 is lost), + 1 = 1e16, - 1e16 = 0, + 4 ones = 4
print("total:", total(v), *(total.lower(d)(v) for d in (single, halves, thirds)))

grid2 = LogicalMesh(  # the same signature: other region points
    Plate,
    {C2C: neighbor_table(C2C, rows)},
    extents={Cell: 8, K: 2},
    regions={West: (1,), East: (2,)},
)
compiled_sharing = program(backend=compiled)(sharing.function)
with contextlib.redirect_stdout(io.StringIO()) as printed:
    compiled_sharing(m0, ones, 0.125, 3)
    compiled_sharing.lower(halves)(m0, ones, 0.125, 3)
    compiled_sharing.lower(thirds)(m0, ones, 0.125, 3)
    on_grid2 = compiled_sharing(
        grid2.field(Cell, data=m0.data), grid2.full(Cell, fill_value=1.0), 0.125, 3
    )
print([line for line in printed.getvalue().splitlines() if "compiling" in line])
print("on grid2, East (cell 2):", on_grid2[2])  # 0.25, as in m

total: 4.0 4.0 4.0 4.0
['[compiled] compiling share']
on grid2, East (cell 2): 0.25


## Local and global

| | 10. Local programs (L1, local semantics) | 13. Global programs (L2, global semantics) |
|---|---|---|
| Source | per rank: `Owned`, `exchange`, `global_sum`, local indices | the whole mesh: `region`, `sum_over` over a `Location`, global indices |
| The mesh | `LocalMesh`es from `decompose`, in `with` blocks | a `LogicalMesh`; its fields carry it |
| Decomposition | in the source | configuration: a `Distribution`, given or derived |
| Exchanges | written by hand (a missing one is an error) | placed by the lowering, once per value |
| Halos | `hops` over `support` | given, or derived from the reads; coverage checked |
| Correctness | by construction of the source | the simulation condition: lowered = direct, bitwise |

Deferred: redundant computation instead of exchanges, programs over several meshes or calling programs, output fields given as arguments, global reductions other than `sum_over` and over several dimensions, a static execution plan.